# Support Articles Engagement Scatter

This notebook compares support articles by **organic traffic** and **average engagement time** so you can quickly spot pages that attract search demand and hold attention.

### Scope
Only pages in these support-content families are included:
- `/get-help/support-toolkit*`
- `/get-help/national-services*`
- `/get-help/hear-from-others*`

### Quick start
1. Run **Setup (run once)**.
2. In **Parameters**, choose your date window and whether to restrict to Australia.
3. Run the query cell.
4. Review the summary table and scatter plot.

**Metrics:** organic sessions, engaged sessions, average engagement time per engaged session  
**Data source:** GA4 BigQuery export (`events_*`)


In [ ]:
#@title Setup (run once)
import sys
import os

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    if not os.path.exists("lla-data"):
        !git clone -q https://github.com/aidoanto/lla-data.git
    repo = os.path.abspath("lla-data")
    if repo not in sys.path:
        sys.path.insert(0, repo)
    !pip install -U -q 'plotly>=6.1.1' 'kaleido>=1.2.0'
else:
    for p in ("..", "../.."):
        ap = os.path.abspath(p)
        if ap not in sys.path:
            sys.path.insert(0, ap)

import pandas as pd
import plotly.express as px
from google.cloud import bigquery

import lifeline_theme
from lla_data import config
from lla_data.bq import (
    build_date_params,
    default_query_window,
    dry_run_bytes,
    get_client,
    load_sql_template,
    run_query,
)

lifeline_theme.inject_fonts()

client = get_client()


In [ ]:
#@title Parameters
DAYS_BACK = 92 #@param {type:"integer"}
AUSTRALIA_ONLY = True #@param {type:"boolean"}
COUNTRY_NAME = "Australia" #@param {type:"string"}
MIN_ORGANIC_SESSIONS = 5 #@param {type:"integer"}

if DAYS_BACK < 1:
    raise ValueError("DAYS_BACK must be at least 1.")

if MIN_ORGANIC_SESSIONS < 0:
    raise ValueError("MIN_ORGANIC_SESSIONS cannot be negative.")

window = default_query_window(days_back=DAYS_BACK)
analysis_start = window.start_date
analysis_end = window.end_date
analysis_window_label = f"{analysis_start:%Y-%m-%d} to {analysis_end:%Y-%m-%d}"
geo_scope_label = f"{COUNTRY_NAME} only" if AUSTRALIA_ONLY else "All countries"

print(f"Project: {config.PROJECT_ID}")
print(f"GA4 dataset: {config.GA4_DATASET}")
print(f"Window: {analysis_start} to {analysis_end}")
print(f"Geo scope: {geo_scope_label}")
print(f"Minimum organic sessions for chart: {MIN_ORGANIC_SESSIONS}")
print("Included page families:")
print("- /get-help/support-toolkit*")
print("- /get-help/national-services*")
print("- /get-help/hear-from-others*")


In [ ]:
query = load_sql_template(
    "sql/notebooks/ga4_support_articles_engagement_scatter.sql",
    project_id=config.PROJECT_ID,
    ga4_dataset=config.GA4_DATASET,
)

params = build_date_params(window) + [
    bigquery.ScalarQueryParameter("australia_only", "BOOL", AUSTRALIA_ONLY),
    bigquery.ScalarQueryParameter("country_name", "STRING", COUNTRY_NAME),
]

estimated_bytes = dry_run_bytes(client, query, params=params)
print(f"Estimated bytes scanned: {estimated_bytes:,}")

df = run_query(client, query, params=params)
print(f"Rows returned: {len(df):,}")

if df.empty:
    raise ValueError("No support-article rows were returned for the selected window and geography filter.")

df.head()


In [ ]:
def classify_family(page_path: str) -> str:
    if page_path.startswith("/get-help/support-toolkit"):
        return "Support toolkit"
    if page_path.startswith("/get-help/national-services"):
        return "National services"
    if page_path.startswith("/get-help/hear-from-others"):
        return "Hear from others"
    return "Other"


numeric_columns = [
    "events",
    "page_views",
    "users",
    "total_sessions",
    "organic_sessions",
    "engaged_sessions",
    "total_engagement_time_seconds",
    "avg_engagement_time_seconds",
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["total_engagement_time_seconds"] = df["total_engagement_time_seconds"].fillna(0.0)
df["avg_engagement_time_seconds"] = df["avg_engagement_time_seconds"].fillna(0.0)
df["content_family"] = df["page_path"].apply(classify_family)

plot_df = df[df["organic_sessions"] >= MIN_ORGANIC_SESSIONS].copy()

if plot_df.empty:
    raise ValueError(
        "Rows were returned, but none met MIN_ORGANIC_SESSIONS. Lower the threshold or widen the date range."
    )

plot_df = plot_df.sort_values(
    ["organic_sessions", "avg_engagement_time_seconds"],
    ascending=[False, False],
).reset_index(drop=True)

excluded_pages = len(df) - len(plot_df)
print(f"Pages in scope before threshold: {len(df):,}")
print(f"Pages shown after threshold: {len(plot_df):,}")
if excluded_pages:
    print(f"Excluded by MIN_ORGANIC_SESSIONS: {excluded_pages:,}")

summary = plot_df[
    [
        "page_path",
        "content_family",
        "organic_sessions",
        "engaged_sessions",
        "total_sessions",
        "avg_engagement_time_seconds",
        "total_engagement_time_seconds",
    ]
].copy()

summary["avg_engagement_time_seconds"] = summary["avg_engagement_time_seconds"].round(1)
summary["total_engagement_time_seconds"] = summary["total_engagement_time_seconds"].round(1)
summary


In [ ]:
color_map = {
    "Support toolkit": lifeline_theme.SUPPORT_BLUE,
    "National services": lifeline_theme.OPTIMISM_ORANGE,
    "Hear from others": lifeline_theme.MINDFUL_MAUVE,
}

fig = px.scatter(
    plot_df,
    x="organic_sessions",
    y="avg_engagement_time_seconds",
    size="engaged_sessions",
    size_max=34,
    color="content_family",
    color_discrete_map=color_map,
    hover_name="page_path",
    hover_data={
        "content_family": True,
        "organic_sessions": ":,",
        "engaged_sessions": ":,",
        "total_sessions": ":,",
        "avg_engagement_time_seconds": ":.1f",
        "total_engagement_time_seconds": ":.1f",
        "page_path": False,
    },
    template="lifeline",
    title=(
        "Support Articles: Organic Traffic vs Average Engagement Time"
        f"<br><span style='font-size:0.8em; font-weight:400;'>{analysis_window_label}</span>"
        f"<br><span style='font-size:0.55em; font-weight:400;'>{geo_scope_label}</span>"
    ),
    labels={
        "organic_sessions": "Organic sessions",
        "avg_engagement_time_seconds": "Avg engagement time per engaged session (seconds)",
        "content_family": "Content family",
    },
)

max_engaged_sessions = plot_df["engaged_sessions"].max()
if max_engaged_sessions > 0:
    sizeref = 2.0 * max_engaged_sessions / (90 ** 2)
    fig.update_traces(
        marker=dict(
            sizemode="area",
            sizeref=sizeref,
            line=dict(width=1, color=lifeline_theme.CALM_BEIGE),
        )
    )

fig.update_layout(height=640, legend_title_text="Content family")
fig.update_xaxes(tickformat=",d")
fig.update_yaxes(ticksuffix="s")
lifeline_theme.add_lifeline_logo(fig)
fig.show()


## Interpretation notes

- Each bubble is a support article page.
- The x-axis uses GA4 `organic_sessions`, which in this repo means distinct `session_start` sessions where medium is `organic`.
- The y-axis uses total GA4 `engagement_time_msec` divided by engaged sessions only, so unengaged visits do not drag the average down.
- Bubble size shows `engaged_sessions`, which helps separate pages that are truly sustaining attention from pages with only a small number of engaged visits.
- If you turn off `AUSTRALIA_ONLY`, the notebook will rerun on all countries using the same page scope.
